In [2]:
"""
GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
=============================================================
GPU-native PTQ using multiple frameworks, since PyTorch's static PTQ
is CPU-only. This script benchmarks three GPU-capable approaches:

  1. ONNX Runtime  — export FP32 → ONNX → INT8 via ORT quantization API
                     (CUDAExecutionProvider for inference)
  2. TensorRT      — export FP32 → ONNX → TRT INT8 engine with calibration
                     (requires tensorrt + pycuda; best GPU throughput)
  3. Torch CUDA    — dynamic PTQ (Linear → qint8) run on CUDA tensors;
                     PT2E (PyTorch 2 Export) + X86InductorQuantizer → torch.compile

FLOPs measurement:
  - fvcore.nn.FlopCountAnalysis for FP32/quantized PyTorch models
  - Manual formula for conv/linear: FLOPs = 2 × Cin × Cout × Kh × Kw × H × W
  - IMPORTANT: FLOPs (operation count) are IDENTICAL for FP32 and INT8 models.
    Quantization does not reduce the number of operations — it changes the
    datatype. The hardware throughput advantage of INT8 is ~4× on Ampere Tensor
    Cores because each SIMD lane processes 4× more INT8 elements per cycle.
    FLOPs and throughput are different quantities and must not be conflated.

Mathematical foundation (same as CPU PTQ, different execution backend):
  Q(x)  = round(x / S) + Z              [quantization]
  x̂     = S · (Q(x) − Z)               [dequantization]
  ε     = x − x̂,  |ε| ≤ S/2           [bounded error]
  S     = (x_max − x_min) / (2^b − 1)  [scale — from calibration data]
  Z     = round(−x_min / S)             [zero-point]

GPU throughput advantage (hardware parallelism, not fewer operations):
  - INT8 Tensor Core GEMM: ~4× throughput vs FP32 on Ampere (A100, RTX 30xx)
  - INT8 Tensor Core GEMM: ~8× throughput vs FP32 on Hopper (H100)
  - Memory bandwidth: ~4× reduction (1 byte INT8 vs 4 bytes FP32 per weight)
  - TensorRT layer fusion: Conv+BN+ReLU → single kernel, fewer memory round-trips
  - ORT CUDAExecutionProvider: uses cuDNN (Conv), cuBLASLt (MatMul), and custom
    kernels depending on op type and graph structure — not all ops use cuDNN INT8.

Output: gpu_ptq_results.json
"""

# ── Imports ────────────────────────────────────────────────────────────────────
import os
import sys
import json
import time
import copy
import math
import tempfile
import warnings
import argparse
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np

warnings.filterwarnings("ignore")

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "ptq_results.json"

BATCH_SIZE     = 128
NUM_WORKERS    = 4
NUM_CLASSES    = 10
CALIB_SIZE     = 1024
CALIB_BATCHES  = 8
INFERENCE_RUNS = 50
INPUT_SHAPE    = (1, 3, 32, 32)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# ── Dependency detection ───────────────────────────────────────────────────────
def _try_import(name: str):
    try:
        import importlib
        return importlib.import_module(name)
    except ImportError:
        return None

torch         = _try_import("torch")
torchvision   = _try_import("torchvision")
ort           = _try_import("onnxruntime")
onnx          = _try_import("onnx")
onnxquant     = _try_import("onnxruntime.quantization")
fvcore        = _try_import("fvcore")
tensorrt      = _try_import("tensorrt")
pycuda        = _try_import("pycuda")
bitsandbytes  = _try_import("bitsandbytes")

INSTALL_COMMANDS = {
    "torch":          "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121",
    "onnx":           "pip install onnx onnxruntime-gpu",
    "onnxruntime":    "pip install onnxruntime-gpu",
    "fvcore":         "pip install fvcore",
    "tensorrt":       "pip install tensorrt",
    "bitsandbytes":   "pip install bitsandbytes",
    "optimum":        "pip install optimum[onnxruntime-gpu]",
}

def install_deps():
    cmds = [
        "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121",
        "pip install onnx onnxruntime-gpu",
        "pip install fvcore",
        "pip install tensorrt pycuda",
        "pip install bitsandbytes",
        "pip install optimum[onnxruntime-gpu]",
    ]
    for cmd in cmds:
        print(f"  $ {cmd}")
        subprocess.run(cmd.split(), check=False)


# ══════════════════════════════════════════════════════════════════════════════
# Model & Data
# ══════════════════════════════════════════════════════════════════════════════
def build_model(num_classes: int = 10):
    import torch.nn as nn
    from torchvision import models
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path: str):
    model = build_model(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    model.eval()
    return model

def get_test_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = datasets.CIFAR10(root="../data", train=False, download=True, transform=tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

def get_calib_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, Subset
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = datasets.CIFAR10(root="../data", train=True, download=True, transform=tf)
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(sub, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# Parameter Counting
# ══════════════════════════════════════════════════════════════════════════════
def count_parameters(model) -> Dict:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "params_total"     : int(total),
        "params_trainable" : int(trainable),
        "params_M"         : round(total / 1e6, 3),
    }


# ══════════════════════════════════════════════════════════════════════════════
# FLOPs Measurement
# ══════════════════════════════════════════════════════════════════════════════
def count_flops_fvcore(model, input_shape=INPUT_SHAPE) -> Dict:
    from fvcore.nn import FlopCountAnalysis, parameter_count
    x = torch.randn(*input_shape)
    flops = FlopCountAnalysis(model, x)
    flops.unsupported_ops_warnings(False)
    total_macs = flops.total()
    params     = parameter_count(model)[""]
    return {
        "total_macs"   : int(total_macs),
        "total_flops"  : int(total_macs * 2),
        "gflops_fp32"  : round(total_macs * 2 / 1e9, 4),
        "params_total" : int(params),
        "params_M"     : round(params / 1e6, 3),
    }

def count_flops_manual(model) -> Dict:
    import torch.nn as nn

    hooks, spatial = [], {}

    def make_hook(name):
        def hook(module, inp, out):
            if hasattr(out, "shape"):
                spatial[name] = tuple(out.shape)
        return hook

    handles = []
    for name, mod in model.named_modules():
        handles.append(mod.register_forward_hook(make_hook(name)))

    x = torch.randn(*INPUT_SHAPE)
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()

    total_flops = 0
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Conv2d):
            shape = spatial.get(name)
            if shape is None:
                continue
            _, cout, hout, wout = shape
            cin  = mod.in_channels
            kh   = mod.kernel_size[0] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            kw   = mod.kernel_size[1] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            macs = cout * (cin // mod.groups) * kh * kw * hout * wout
            total_flops += 2 * macs
        elif isinstance(mod, nn.Linear):
            total_flops += 2 * mod.in_features * mod.out_features

    params_total = sum(p.numel() for p in model.parameters())
    return {
        "total_flops" : total_flops,
        "gflops_fp32" : round(total_flops / 1e9, 4),
        "params_total": int(params_total),
        "params_M"    : round(params_total / 1e6, 3),
    }

def measure_gpu_throughput(model, device, n: int = INFERENCE_RUNS) -> Dict:
    """
    Returns the inference_ms block matching the example JSON format:
      single_img_gpu_ms, batch128_gpu_ms, per_img_gpu_ms,
      throughput_imgs_sec, timing_method
    """
    model = model.to(device).eval()

    # ── Batch timing (CUDA events) ────────────────────────────────────────────
    dummy_batch = torch.randn(BATCH_SIZE, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(10):
            model(dummy_batch)
    torch.cuda.synchronize(device)

    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev   = torch.cuda.Event(enable_timing=True)
    batch_timings = []
    with torch.no_grad():
        for _ in range(n):
            start_ev.record()
            model(dummy_batch)
            end_ev.record()
            torch.cuda.synchronize()
            batch_timings.append(start_ev.elapsed_time(end_ev))

    # ── Single-image timing (CUDA events) ─────────────────────────────────────
    dummy_single = torch.randn(1, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(10):
            model(dummy_single)
    torch.cuda.synchronize(device)

    single_timings = []
    with torch.no_grad():
        for _ in range(n):
            start_ev.record()
            model(dummy_single)
            end_ev.record()
            torch.cuda.synchronize()
            single_timings.append(start_ev.elapsed_time(end_ev))

    batch_ms  = float(np.mean(batch_timings))
    single_ms = float(np.mean(single_timings))

    return {
        "single_img_gpu_ms" : round(single_ms, 4),
        "batch128_gpu_ms"   : round(batch_ms, 4),
        "per_img_gpu_ms"    : round(batch_ms / BATCH_SIZE, 4),
        "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
        "timing_method"     : "CUDA events + torch.cuda.synchronize()",
    }

def measure_ort_throughput(session, n: int = INFERENCE_RUNS) -> Dict:
    """ORT wall-clock timing, formatted to match example JSON."""
    import numpy as np
    input_name  = session.get_inputs()[0].name
    dummy_batch = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
    dummy_single = np.random.randn(1, 3, 32, 32).astype(np.float32)

    # Warmup
    for _ in range(5):
        session.run(None, {input_name: dummy_batch})

    # Batch
    t0 = time.perf_counter()
    for _ in range(n):
        session.run(None, {input_name: dummy_batch})
    batch_ms = (time.perf_counter() - t0) / n * 1000

    # Single
    for _ in range(5):
        session.run(None, {input_name: dummy_single})
    t0 = time.perf_counter()
    for _ in range(n):
        session.run(None, {input_name: dummy_single})
    single_ms = (time.perf_counter() - t0) / n * 1000

    return {
        "single_img_gpu_ms" : round(single_ms, 4),
        "batch128_gpu_ms"   : round(batch_ms, 4),
        "per_img_gpu_ms"    : round(batch_ms / BATCH_SIZE, 4),
        "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
        "timing_method"     : "wall-clock (time.perf_counter)",
    }

def disk_size_mb(obj) -> float:
    if isinstance(obj, str):
        return os.path.getsize(obj) / 1024 ** 2
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(obj, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)


# ══════════════════════════════════════════════════════════════════════════════
# Build a compact result entry (matches example JSON schema)
# ══════════════════════════════════════════════════════════════════════════════
def build_result_entry(
    metrics: Dict,          # accuracy, precision, recall, f1
    params_total: int,
    size_mb: float,
    flops_G: float,
    inference_ms: Dict,     # single_img_gpu_ms, batch128_gpu_ms, per_img_gpu_ms,
                            # throughput_imgs_sec, timing_method
) -> Dict:
    """
    Returns a flat dict matching the example JSON schema exactly:
      accuracy, precision, recall, f1,
      params, size_mb, flops_G, inference_ms
    No nested quantization_math, no description blobs, no flops_note.
    """
    return {
        "accuracy"    : round(metrics["accuracy"],  6),
        "precision"   : round(metrics["precision"], 6),
        "recall"      : round(metrics["recall"],    6),
        "f1"          : round(metrics["f1"],        6),
        "params"      : int(params_total),
        "size_mb"     : round(size_mb, 6) if size_mb is not None else None,
        "flops_G"     : round(flops_G, 9),
        "inference_ms": inference_ms,
    }


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate_torch(model, loader, device) -> Dict:
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    model = model.to(device).eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(device)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def evaluate_onnx(session, loader) -> Dict:
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    input_name = session.get_inputs()[0].name
    preds, labels = [], []
    for inputs, lbls in loader:
        out = session.run(None, {input_name: inputs.numpy()})[0]
        preds.extend(np.argmax(out, axis=1))
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


# ══════════════════════════════════════════════════════════════════════════════
# ONNX Export
# ══════════════════════════════════════════════════════════════════════════════
def export_to_onnx(model, output_path: str, opset: int = 17) -> str:
    dummy = torch.randn(*INPUT_SHAPE)
    torch.onnx.export(
        model.cpu().eval(),
        dummy,
        output_path,
        opset_version        = opset,
        input_names          = ["input"],
        output_names         = ["output"],
        dynamic_axes         = {"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        do_constant_folding  = True,
        export_params        = True,
    )
    print(f"        ✓ ONNX exported → {output_path}  "
          f"({os.path.getsize(output_path)/1024**2:.2f} MB, opset={opset})")
    return output_path


# ══════════════════════════════════════════════════════════════════════════════
# 1. ONNX Runtime — INT8 Static Quantization (GPU inference)
# ══════════════════════════════════════════════════════════════════════════════
def run_onnx_ptq(fp32_model, test_loader, calib_loader,
                 flops_G: float, params_total: int,
                 device: str) -> Dict:
    import onnxruntime
    from onnxruntime.quantization import (
        quantize_static, CalibrationDataReader,
        QuantType, QuantFormat, CalibrationMethod
    )

    print("\n  [1/3] ONNX Runtime — Static INT8 PTQ")

    fp32_onnx = "resnet50_fp32.onnx"
    export_to_onnx(fp32_model, fp32_onnx)

    class CIFARCalibReader(CalibrationDataReader):
        def __init__(self, loader, max_batches=CALIB_BATCHES):
            self.data    = iter(loader)
            self.batches = 0
            self.max     = max_batches

        def get_next(self):
            if self.batches >= self.max:
                return None
            try:
                imgs, _ = next(self.data)
                self.batches += 1
                return {"input": np.ascontiguousarray(imgs.numpy(), dtype=np.float32)}
            except StopIteration:
                return None

    results = {}
    calibration_methods = {
        "ort_minmax"     : CalibrationMethod.MinMax,
        "ort_entropy"    : CalibrationMethod.Entropy,
        "ort_percentile" : CalibrationMethod.Percentile,
    }

    for key, calib_method in calibration_methods.items():
        calib_label = key.replace("ort_", "").capitalize()
        print(f"        Calibration: {calib_label:12s}", end="", flush=True)
        int8_onnx = f"resnet50_int8_{calib_label.lower()}.onnx"
        try:
            quantize_static(
                model_input             = fp32_onnx,
                model_output            = int8_onnx,
                calibration_data_reader = CIFARCalibReader(calib_loader),
                quant_format            = QuantFormat.QDQ,
                per_channel             = True,
                reduce_range            = False,
                weight_type             = QuantType.QInt8,
                activation_type         = QuantType.QUInt8,
                calibrate_method        = calib_method,
                extra_options           = {
                    "EnableSubgraph"                 : True,
                    "MatMulConstBOnly"               : False,
                    "AddQDQPairToWeight"             : True,
                    "QuantizeAllFixedZeroPointTensors": True,
                },
            )

            providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                         if "CUDAExecutionProvider" in onnxruntime.get_available_providers()
                         else ["CPUExecutionProvider"])
            sess_opts = onnxruntime.SessionOptions()
            sess_opts.graph_optimization_level = (
                onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
            )
            session = onnxruntime.InferenceSession(int8_onnx, sess_opts,
                                                    providers=providers)

            metrics      = evaluate_onnx(session, test_loader)
            size_mb      = disk_size_mb(int8_onnx)
            inference_ms = measure_ort_throughput(session)

            results[key] = build_result_entry(
                metrics      = metrics,
                params_total = params_total,
                size_mb      = size_mb,
                flops_G      = flops_G,
                inference_ms = inference_ms,
            )

            print(f" → Acc: {metrics['accuracy']:.4f}  "
                  f"Disk: {size_mb:.2f} MB  "
                  f"Lat: {inference_ms['batch128_gpu_ms']:.1f} ms "
                  f"({session.get_providers()[0]})")

            os.makedirs("__1__saved_models_PTQ", exist_ok=True)
            import shutil
            shutil.copy(int8_onnx, f"__1__saved_models_PTQ/onnx_int8_{calib_label.lower()}.onnx")
            print(f"        ✓ Saved → __1__saved_models_PTQ/onnx_int8_{calib_label.lower()}.onnx")

        except Exception as e:
            print(f" → FAILED: {e}")
            results[key] = None

        finally:
            if os.path.exists(int8_onnx):
                os.remove(int8_onnx)

    if os.path.exists(fp32_onnx):
        os.remove(fp32_onnx)

    return results


# ══════════════════════════════════════════════════════════════════════════════
# 2. TensorRT — INT8 PTQ with Calibration Cache
# ══════════════════════════════════════════════════════════════════════════════
def run_tensorrt_ptq(fp32_model, test_loader, calib_loader,
                     flops_G: float, params_total: int,
                     device: str) -> Optional[Dict]:
    print("\n  [2/3] TensorRT — INT8 PTQ")
    try:
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit
    except ImportError as e:
        print(f"        ⚠ TensorRT/pycuda not available: {e}")
        return None

    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    fp32_onnx  = "resnet50_trt_fp32.onnx"

    try:
        export_to_onnx(fp32_model, fp32_onnx)

        class CIFARCalibrator(trt.IInt8EntropyCalibrator2):
            def __init__(self, loader, cache_file="trt_calib.cache"):
                super().__init__()
                self.loader     = iter(loader)
                self.batches    = 0
                self.cache_file = cache_file
                batch, _ = next(iter(loader))
                self.batch_size = batch.shape[0]
                self.device_mem = cuda.mem_alloc(int(np.prod(batch.shape) * 4))

            def get_batch_size(self):
                return self.batch_size

            def get_batch(self, names):
                if self.batches >= CALIB_BATCHES:
                    return None
                try:
                    imgs, _ = next(self.loader)
                    cuda.memcpy_htod(self.device_mem, imgs.numpy().astype(np.float32))
                    self.batches += 1
                    return [int(self.device_mem)]
                except StopIteration:
                    return None

            def read_calibration_cache(self):
                if os.path.exists(self.cache_file):
                    with open(self.cache_file, "rb") as f:
                        return f.read()
                return None

            def write_calibration_cache(self, cache):
                with open(self.cache_file, "wb") as f:
                    f.write(cache)

        builder = trt.Builder(TRT_LOGGER)
        network = builder.create_network(
            1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        )
        parser = trt.OnnxParser(network, TRT_LOGGER)
        with open(fp32_onnx, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError(f"ONNX parse error: {parser.get_error(0)}")

        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
        config.set_flag(trt.BuilderFlag.INT8)
        config.set_flag(trt.BuilderFlag.FP16)
        config.int8_calibrator = CIFARCalibrator(calib_loader)

        profile = builder.create_optimization_profile()
        profile.set_shape("input",
                           min=(1, 3, 32, 32),
                           opt=(BATCH_SIZE, 3, 32, 32),
                           max=(BATCH_SIZE * 2, 3, 32, 32))
        config.add_optimization_profile(profile)

        print("        Building TRT INT8 engine...")
        serialized = builder.build_serialized_network(network, config)

        trt_path = "resnet50_int8.trt"
        with open(trt_path, "wb") as f:
            f.write(serialized)

        runtime = trt.Runtime(TRT_LOGGER)
        engine  = runtime.deserialize_cuda_engine(serialized)
        context = engine.create_execution_context()
        context.set_input_shape("input", (BATCH_SIZE, 3, 32, 32))

        d_input  = cuda.mem_alloc(BATCH_SIZE * 3 * 32 * 32 * 4)
        d_output = cuda.mem_alloc(BATCH_SIZE * NUM_CLASSES * 4)
        stream   = cuda.Stream()

        def infer_trt(x_np):
            cuda.memcpy_htod_async(d_input, x_np.astype(np.float32), stream)
            context.execute_async_v2(
                bindings=[int(d_input), int(d_output)], stream_handle=stream.handle
            )
            out = np.empty((BATCH_SIZE, NUM_CLASSES), dtype=np.float32)
            cuda.memcpy_dtoh_async(out, d_output, stream)
            stream.synchronize()
            return out

        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
        preds, labels = [], []
        for inputs, lbls in test_loader:
            out = infer_trt(inputs.numpy())
            preds.extend(np.argmax(out, 1))
            labels.extend(lbls.numpy())
        y_pred, y_true = np.array(preds), np.array(labels)
        metrics = {
            "accuracy" : float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
            "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        }

        # ── TRT latency (CUDA events via pycuda) ──────────────────────────────
        dummy_np  = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
        dummy_s   = np.random.randn(1, 3, 32, 32).astype(np.float32)
        start_ev  = cuda.Event()
        end_ev    = cuda.Event()

        # Single-image engine (rebuild context for batch=1)
        context1 = engine.create_execution_context()
        context1.set_input_shape("input", (1, 3, 32, 32))
        d_in1  = cuda.mem_alloc(1 * 3 * 32 * 32 * 4)
        d_out1 = cuda.mem_alloc(1 * NUM_CLASSES * 4)

        def infer_trt_single(x_np):
            cuda.memcpy_htod_async(d_in1, x_np.astype(np.float32), stream)
            context1.execute_async_v2(
                bindings=[int(d_in1), int(d_out1)], stream_handle=stream.handle
            )
            out = np.empty((1, NUM_CLASSES), dtype=np.float32)
            cuda.memcpy_dtoh_async(out, d_out1, stream)
            stream.synchronize()
            return out

        for _ in range(5):
            infer_trt(dummy_np)
            infer_trt_single(dummy_s)

        batch_times, single_times = [], []
        for _ in range(INFERENCE_RUNS):
            start_ev.record()
            infer_trt(dummy_np)
            end_ev.record()
            end_ev.synchronize()
            batch_times.append(start_ev.time_till(end_ev))

            start_ev.record()
            infer_trt_single(dummy_s)
            end_ev.record()
            end_ev.synchronize()
            single_times.append(start_ev.time_till(end_ev))

        batch_ms  = float(np.mean(batch_times))
        single_ms = float(np.mean(single_times))

        inference_ms = {
            "single_img_gpu_ms"  : round(single_ms, 4),
            "batch128_gpu_ms"    : round(batch_ms, 4),
            "per_img_gpu_ms"     : round(batch_ms / BATCH_SIZE, 4),
            "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
            "timing_method"      : "CUDA events + torch.cuda.synchronize()",
        }

        size_mb = disk_size_mb(trt_path)
        result  = build_result_entry(
            metrics      = metrics,
            params_total = params_total,
            size_mb      = size_mb,
            flops_G      = flops_G,
            inference_ms = inference_ms,
        )

        print(f"        → Acc: {metrics['accuracy']:.4f}  "
              f"Disk: {size_mb:.2f} MB  GPU: {batch_ms:.1f} ms")

        os.makedirs("__1__saved_models_PTQ", exist_ok=True)
        import shutil
        shutil.copy(trt_path, "__1__saved_models_PTQ/tensorrt_int8.trt")
        print(f"        ✓ Saved → __1__saved_models_PTQ/tensorrt_int8.trt")

        return result

    except Exception as e:
        print(f"        → FAILED: {e}")
        return None
    finally:
        for f in [fp32_onnx, "trt_calib.cache", "resnet50_int8.trt"]:
            if os.path.exists(f):
                os.remove(f)


# ══════════════════════════════════════════════════════════════════════════════
# 3. PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8
# ══════════════════════════════════════════════════════════════════════════════
def _resolve_pt2e_imports():
    errors = []

    prepare_pt2e = convert_pt2e = None
    for mod_path in [
        "torch.ao.quantization.quantize_pt2e",
        "torch.quantization.quantize_pt2e",
    ]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            prepare_pt2e = m.prepare_pt2e
            convert_pt2e = m.convert_pt2e
            break
        except (ImportError, AttributeError) as e:
            errors.append(f"  {mod_path}: {e}")

    if prepare_pt2e is None:
        raise ImportError(
            "Cannot find prepare_pt2e / convert_pt2e. Tried:\n" + "\n".join(errors) +
            "\nRequires PyTorch >= 2.3."
        )

    Quantizer = get_config = None
    for mod_path, cls, cfg in [
        (
            "torch.ao.quantization.quantizer.x86_inductor_quantizer",
            "X86InductorQuantizer",
            "get_default_x86_inductor_quantization_config",
        ),
    ]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            Quantizer  = getattr(m, cls)
            get_config = getattr(m, cfg)
            break
        except (ImportError, AttributeError) as e:
            errors.append(f"  {mod_path}.{cls}: {e}")

    if Quantizer is None:
        raise ImportError(
            "Cannot find X86InductorQuantizer. Tried:\n" + "\n".join(errors)
        )

    export_fn = None
    try:
        from torch.export import export as _torch_export
        def export_fn(model, example_args):
            ep = _torch_export(model, example_args)
            return ep.module()
    except (ImportError, AttributeError) as e:
        errors.append(f"  torch.export.export: {e}")

    if export_fn is None:
        try:
            from torch._export import capture_pre_autograd_graph as _cap
            def export_fn(model, example_args):
                return _cap(model, example_args)
        except (ImportError, AttributeError) as e:
            errors.append(f"  torch._export.capture_pre_autograd_graph: {e}")

    if export_fn is None:
        raise ImportError("Cannot find a graph export function. Tried:\n" + "\n".join(errors))

    return prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn


def run_torch_cuda_ptq(fp32_model, test_loader, calib_loader,
                        flops_G: float, params_total: int,
                        device: str) -> Dict:
    print("\n  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8")
    results = {}
    import torch.nn as nn

    # ── 3a. Dynamic PTQ on CUDA ────────────────────────────────────────────────
    print("        3a. Dynamic PTQ (Linear → qint8) on CUDA")
    try:
        q_dyn = torch.quantization.quantize_dynamic(
            copy.deepcopy(fp32_model), {nn.Linear}, dtype=torch.qint8
        )
        q_dyn = q_dyn.to(device).eval()

        os.makedirs("__1__saved_models_PTQ", exist_ok=True)
        torch.save(q_dyn.cpu().state_dict(), "__1__saved_models_PTQ/dynamic_ptq_int8.pth")
        print(f"        ✓ Saved → __1__saved_models_PTQ/dynamic_ptq_int8.pth")
        q_dyn = q_dyn.to(device)

        metrics      = evaluate_torch(q_dyn, test_loader, device)
        inference_ms = measure_gpu_throughput(q_dyn, device)
        size_mb      = disk_size_mb(q_dyn.cpu())

        results["dynamic_ptq"] = build_result_entry(
            metrics      = metrics,
            params_total = params_total,
            size_mb      = size_mb,
            flops_G      = flops_G,
            inference_ms = inference_ms,
        )
        print(f"           → Acc: {metrics['accuracy']:.4f}  "
              f"GPU: {inference_ms['batch128_gpu_ms']:.1f} ms  Disk: {size_mb:.2f} MB")

    except Exception as e:
        print(f"           → FAILED: {e}")
        results["dynamic_ptq"] = None

    # ── 3b. PT2E + X86InductorQuantizer → torch.compile ──────────────────────
    print("        3b. PT2E (torch.export + X86InductorQuantizer) → torch.compile")
    try:
        prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn = (
            _resolve_pt2e_imports()
        )
        print(f"           PT2E imports resolved (torch {torch.__version__})")

        example_args = (torch.randn(1, 3, 32, 32, device=device),)
        m = copy.deepcopy(fp32_model).to(device).eval()
        exported = export_fn(m, example_args)

        quantizer = Quantizer()
        quantizer.set_global(get_config())
        prepared = prepare_pt2e(exported, quantizer)

        prepared.eval()
        with torch.no_grad():
            for i, (imgs, _) in enumerate(calib_loader):
                prepared(imgs.to(device))
                if i + 1 >= CALIB_BATCHES:
                    break

        q_pt2e     = convert_pt2e(prepared)
        q_compiled = torch.compile(q_pt2e, mode="max-autotune", backend="inductor")

        os.makedirs("__1__saved_models_PTQ", exist_ok=True)
        try:
            torch.save(q_pt2e.state_dict(), "__1__saved_models_PTQ/pt2e_int8.pth")
            print(f"        ✓ Saved → __1__saved_models_PTQ/pt2e_int8.pth")
        except Exception as se:
            print(f"        ⚠ PT2E save failed (expected): {se}")

        metrics      = evaluate_torch(q_compiled, test_loader, device)
        inference_ms = measure_gpu_throughput(q_compiled, device)

        results["pt2e_inductor"] = build_result_entry(
            metrics      = metrics,
            params_total = params_total,
            size_mb      = None,   # compiled model not trivially serialisable
            flops_G      = flops_G,
            inference_ms = inference_ms,
        )
        print(f"           → Acc: {metrics['accuracy']:.4f}  "
              f"GPU: {inference_ms['batch128_gpu_ms']:.1f} ms")

    except ImportError as e:
        print(f"           → SKIPPED (PT2E unavailable): {e}")
        results["pt2e_inductor"] = None
    except Exception as e:
        print(f"           → FAILED: {e}")
        results["pt2e_inductor"] = None

    return results


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    parser = argparse.ArgumentParser(description="GPU PTQ for ResNet-50/CIFAR-10")
    parser.add_argument("--install-deps", action="store_true",
                        help="Install required GPU packages and exit")
    parser.add_argument("--device", default="cuda",
                        help="Target device: cuda, cuda:0, cuda:1, cpu")
    args, _ = parser.parse_known_args()

    if args.install_deps:
        install_deps()
        return

    # ── Dependency check ───────────────────────────────────────────────────────
    missing = []
    if torch is None:
        missing.append("torch  →  pip install torch --index-url https://download.pytorch.org/whl/cu121")
    if onnx is None:
        missing.append("onnx   →  pip install onnx")
    if ort is None:
        missing.append("onnxruntime-gpu  →  pip install onnxruntime-gpu")
    if missing:
        print("\n⚠  Missing required packages:")
        for m in missing:
            print(f"   {m}")
        print("\nRun:  python gpu_ptq.py --install-deps")
        sys.exit(1)

    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*65}")
    print("  GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10")
    print(f"  Device    : {device}")
    if torch.cuda.is_available():
        print(f"  GPU       : {torch.cuda.get_device_name(device)}")
        print(f"  CUDA      : {torch.version.cuda}")
        print(f"  cuDNN     : {torch.backends.cudnn.version()}")
    print("  Frameworks: ONNX Runtime GPU  |  TensorRT  |  PyTorch PT2E")
    print(f"{'='*65}")

    # ── Load baseline ──────────────────────────────────────────────────────────
    if not os.path.exists(BASELINE_MODEL_PATH):
        print(f"\n✗ Baseline model not found: {BASELINE_MODEL_PATH}")
        print("  Run the baseline training script first.")
        sys.exit(1)

    with open(BASELINE_METRICS_PATH) as f:
        baseline_metrics = json.load(f)

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    test_loader  = get_test_loader()
    calib_loader = get_calib_loader()

    # ── FLOPs (computed once; identical for FP32 and INT8) ────────────────────
    print("\n  Computing FLOPs...")
    flops_info = {}
    if fvcore is not None:
        try:
            flops_info = count_flops_fvcore(fp32_model.cpu())
        except Exception:
            flops_info = count_flops_manual(fp32_model.cpu())
    else:
        flops_info = count_flops_manual(fp32_model.cpu())

    flops_G      = flops_info.get("gflops_fp32", 0)
    params_total = flops_info.get("params_total",
                                   sum(p.numel() for p in fp32_model.parameters()))
    print(f"  FLOPs: {flops_G:.3f} GFLOPs | Params: {params_total/1e6:.2f}M")

    # ── Run all backends ───────────────────────────────────────────────────────
    ort_results = run_onnx_ptq(
        fp32_model, test_loader, calib_loader, flops_G, params_total, device
    )
    trt_result  = run_tensorrt_ptq(
        fp32_model, test_loader, calib_loader, flops_G, params_total, device
    )
    torch_results = run_torch_cuda_ptq(
        fp32_model, test_loader, calib_loader, flops_G, params_total, device
    )

    # ── Assemble final JSON (flat dict-of-dicts, keyed by method name) ─────────
    output = {}

    # ONNX Runtime variants
    for key, val in ort_results.items():
        if val is not None:
            output[key] = val

    # TensorRT
    if trt_result is not None:
        output["tensorrt_int8"] = trt_result

    # PyTorch CUDA dynamic + PT2E
    for key, val in torch_results.items():
        if val is not None:
            output[key] = val

    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)

    # ── Console summary ────────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  ✓ Saved → {OUTPUT_JSON}")
    print(f"\n  {'Method':<22} {'Acc':>7} {'F1':>7} {'Size(MB)':>9} "
          f"{'Batch(ms)':>10} {'Imgs/s':>8}")
    print(f"  {'-'*65}")
    for key, val in output.items():
        if val is None:
            continue
        size_str = f"{val['size_mb']:.2f}" if val.get("size_mb") is not None else "  N/A"
        lat      = val["inference_ms"]["batch128_gpu_ms"]
        tput     = val["inference_ms"]["throughput_imgs_sec"]
        print(f"  {key:<22} {val['accuracy']:.4f}  {val['f1']:.4f}  "
              f"{size_str:>9}  {lat:>10.1f}  {tput:>8.1f}")
    print(f"{'='*65}\n")


if __name__ == "__main__":
    main()


  GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
  Device    : cuda
  GPU       : NVIDIA GeForce RTX 5070 Laptop GPU
  CUDA      : 12.8
  cuDNN     : 92000
  Frameworks: ONNX Runtime GPU  |  TensorRT  |  PyTorch PT2E


W0429 23:55:37.101000 3796 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features



  Computing FLOPs...
  FLOPs: 2.596 GFLOPs | Params: 23.52M

  [1/3] ONNX Runtime — Static INT8 PTQ
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


Traceback (most recent call last):
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnx\version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: D:\a\onnx\onnx\onnx/version

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
        ✓ ONNX exported → resnet50_fp32.onnx  (0.22 MB, opset=17)
        Calibration: Minmax      

 → Acc: 0.9319  Disk: 90.18 MB  Lat: 63.9 ms (CUDAExecutionProvider)
        ✓ Saved → __1__saved_models_PTQ/onnx_int8_minmax.onnx
        Calibration: Entropy     

Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 122
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128


 → Acc: 0.9319  Disk: 90.18 MB  Lat: 68.1 ms (CUDAExecutionProvider)
        ✓ Saved → __1__saved_models_PTQ/onnx_int8_entropy.onnx
        Calibration: Percentile  

Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 122
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)


 → Acc: 0.9316  Disk: 90.18 MB  Lat: 66.6 ms (CUDAExecutionProvider)
        ✓ Saved → __1__saved_models_PTQ/onnx_int8_percentile.onnx

  [2/3] TensorRT — INT8 PTQ
        ⚠ TensorRT/pycuda not available: No module named 'tensorrt'

  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8
        3a. Dynamic PTQ (Linear → qint8) on CUDA
        ✓ Saved → __1__saved_models_PTQ/dynamic_ptq_int8.pth
           → FAILED: Could not run 'quantized::linear_dynamic' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::linear_dynamic' is only available for these backends: [CPU, Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, Autograd